[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/37_uncertainty_and_calibration/01_calibration_reliability_brier.ipynb)

# Bài 01 — Calibration: reliability diagram, ECE, Brier

**🎯 Mục tiêu**
- **Calibration** là gì và vì sao quan trọng tách biệt với khả năng xếp hạng (AUC).
- Cài **reliability diagram** và **ECE** from scratch; hiểu "sàn nhiễu" của ECE với mẫu hữu hạn.
- **Scoring rule** đúng đắn: Brier, log-loss; **phân rã Brier** (Murphy) thành reliability, resolution, uncertainty.
- So sánh calibration của các họ mô hình trên dữ liệu có **xác suất thật đã biết**.
- Xác suất sai lệch dẫn tới **quyết định tốn kém** dù AUC tốt.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/37_uncertainty_and_calibration

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_customer_churn
from ml_course.visualization import set_style

set_style()
warnings.filterwarnings("ignore", category=UserWarning)

## 1. Calibration là gì?

Một mô hình **calibrated** nếu trong số các trường hợp nó dự đoán xác suất $p$, tỉ lệ thật sự xảy ra đúng bằng $p$: $\mathbb P(Y = 1\mid \hat p = p) = p$. "70% khả năng mưa" nghĩa là: trong mọi ngày dự báo 70%, khoảng 70% ngày có mưa.

**Xếp hạng ≠ calibration**: AUC chỉ phụ thuộc **thứ tự** các điểm số; bất kỳ biến đổi đơn điệu nào ($p \mapsto p^3$) giữ nguyên AUC nhưng làm hỏng xác suất. Cần xác suất đúng khi:
- ra quyết định theo chi phí (giữ chân khách hàng có lợi không?), định giá rủi ro, bảo hiểm;
- truyền đạt rủi ro cho con người (bác sĩ, khách hàng);
- kết hợp nhiều mô hình, hoặc đưa xác suất vào hệ thống khác.

## 2. Dữ liệu có xác suất thật

Dữ liệu rời bỏ khách hàng (chương 7, 36) sinh từ logit thật → biết **xác suất thật** $p^*(x)$ của từng khách hàng — điều không bao giờ có với dữ liệu thật, cho phép đo calibration "chuẩn" bằng $\text{MSE}(\hat p, p^*)$.

In [ ]:
raw = make_customer_churn(30000, seed=0)
X = pd.DataFrame({
    "tenure": raw.tenure_months, "monthly": raw.monthly_charges, "total": raw.total_charges,
    "support_calls": raw.support_calls, "month_to_month": (raw.contract == "month-to-month").astype(int),
    "two_year": (raw.contract == "two_year").astype(int), "fiber": (raw.internet_service == "fiber").astype(int),
    "ewallet": (raw.payment_method == "e-wallet").astype(int), "senior": raw.senior.astype(int), "n_products": raw.n_products,
}).astype(float)
true_logit = (-1.6 + 1.3 * X.month_to_month - 0.5 * X.two_year - 0.035 * X.tenure + 0.018 * (X.monthly - 70) + 0.38 * X.support_calls
              + 0.45 * X.fiber + 0.35 * X.ewallet + 0.3 * X.senior - 0.2 * (X.n_products - 1))
p_true = 1 / (1 + np.exp(-true_logit.values))
y = raw.churn.values
idx_train, idx_rest = train_test_split(np.arange(len(X)), test_size=0.4, random_state=0)
idx_cal, idx_test = train_test_split(idx_rest, test_size=0.5, random_state=0)       # tập calibration dùng ở bài 02
print(f"train {len(idx_train):,} | calibration {len(idx_cal):,} | test {len(idx_test):,} | tỉ lệ rời bỏ {y.mean():.1%}")

## 3. Reliability diagram và ECE

Chia dự đoán thành các **khoảng** (bin) theo $\hat p$; trong mỗi khoảng so **trung bình dự đoán** với **tỉ lệ thật**. Mô hình calibrated nằm trên đường chéo.

**ECE** (Expected Calibration Error) = trung bình có trọng số (theo số mẫu) của |tỉ lệ thật − dự đoán trung bình| qua các khoảng.

In [ ]:
def reliability(p, y, n_bins=10):
    """Trả về DataFrame theo bin: dự đoán TB, tỉ lệ thật, số mẫu (bin chia đều trên [0, 1])."""
    bins = np.clip((p * n_bins).astype(int), 0, n_bins - 1)
    rows = [(p[bins == b].mean(), y[bins == b].mean(), (bins == b).sum()) for b in range(n_bins) if (bins == b).any()]
    return pd.DataFrame(rows, columns=["mean_pred", "frac_pos", "count"])


def ece(p, y, n_bins=10):
    r = reliability(p, y, n_bins)
    return np.sum(r["count"] * (r["frac_pos"] - r["mean_pred"]).abs()) / r["count"].sum()


y_test, pt_test = y[idx_test], p_true[idx_test]
print(f"ECE của chính XÁC SUẤT THẬT trên {len(idx_test):,} mẫu test: {ece(pt_test, y_test):.4f}")
sim = [ece(pt_test, (np.random.default_rng(s).random(len(pt_test)) < pt_test).astype(int)) for s in range(200)]
print(f"ECE của xác suất thật qua 200 lần mô phỏng nhãn: trung bình {np.mean(sim):.4f}, 95% ≤ {np.quantile(sim, 0.95):.4f}")

Ngay cả **xác suất thật** cũng có ECE khác 0: nhãn là ngẫu nhiên, mỗi bin chỉ có hữu hạn mẫu → tỉ lệ thật dao động quanh $p$. ECE nhỏ hơn mức "sàn nhiễu" này không có ý nghĩa; muốn so sánh, cần đủ mẫu, và đừng đọc chênh lệch ECE vài phần nghìn như thật. (ECE còn phụ thuộc số bin và cách chia bin.)

## 4. Calibration của các họ mô hình

In [ ]:
models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Naive Bayes": GaussianNB(),
    "Random forest": RandomForestClassifier(300, min_samples_leaf=3, random_state=0, n_jobs=-1),
    "Gradient boosting": HistGradientBoostingClassifier(random_state=0),
    "MLP (overfit)": make_pipeline(StandardScaler(), MLPClassifier((128, 128), alpha=0, max_iter=500, random_state=0)),
}
probs, rows = {}, []
for name, model in models.items():
    t0 = time.perf_counter()
    fit_idx = idx_train[:3000] if "MLP" in name else idx_train                   # MLP: ít dữ liệu, không regularization → overfit
    model.fit(X.iloc[fit_idx], y[fit_idx])
    p = model.predict_proba(X.iloc[idx_test])[:, 1]
    probs[name] = p
    rows.append({"mô hình": name, "ROC-AUC": roc_auc_score(y_test, p), "Brier": brier_score_loss(y_test, p), "log-loss": log_loss(y_test, p),
                 "ECE": ece(p, y_test), "MSE với p thật": np.mean((p - pt_test) ** 2), "thời gian (s)": time.perf_counter() - t0})
rows.append({"mô hình": "xác suất thật", "ROC-AUC": roc_auc_score(y_test, pt_test), "Brier": brier_score_loss(y_test, pt_test),
             "log-loss": log_loss(y_test, pt_test), "ECE": ece(pt_test, y_test), "MSE với p thật": 0.0, "thời gian (s)": 0})
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, p in probs.items():
    r = reliability(p, y_test)
    axes[0].plot(r.mean_pred, r.frac_pos, "o-", label=name)
axes[0].plot([0, 1], [0, 1], "k:", label="calibrated hoàn hảo")
axes[0].set_xlabel("xác suất dự đoán (trung bình bin)")
axes[0].set_ylabel("tỉ lệ rời bỏ thật trong bin")
axes[0].legend(fontsize=8)
for name in ["Logistic regression", "Naive Bayes", "MLP (overfit)"]:
    axes[1].hist(probs[name], bins=40, histtype="step", density=True, label=name)
axes[1].set_xlabel("xác suất dự đoán")
axes[1].set_title("phân phối xác suất dự đoán")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

- **Logistic regression** (đúng dạng với dữ liệu): calibrated gần như hoàn hảo — ECE ngang "sàn nhiễu", MSE với $p^*$ gần 0.
- **Naive Bayes**: giả định độc lập sai (total phụ thuộc tenure và monthly — đếm trùng bằng chứng) → xác suất bị đẩy về 0 và 1, **quá tự tin**.
- **MLP overfit** (ít dữ liệu, không regularization): học thuộc tập train → quá tự tin trên dữ liệu mới, xác suất dồn về hai đầu; đây là hiện tượng chung của mạng nơ-ron hiện đại (Guo et al., 2017).
- **Random forest / gradient boosting** trên dữ liệu này khá calibrated; nói chung rừng có xu hướng **kém tự tin** ở hai đầu (trung bình nhiều cây kéo về giữa), boosting thường ổn nhưng có thể quá tự tin khi rất nhiều vòng.

Chú ý cột ROC-AUC: Naive Bayes xếp hạng gần bằng gradient boosting nhưng xác suất sai lệch hẳn — AUC không phát hiện được.

## 5. Scoring rule đúng đắn và phân rã Brier

**Brier** $= \frac1n\sum(\hat p_i - y_i)^2$ và **log-loss** là các **proper scoring rule**: kỳ vọng của chúng đạt cực tiểu khi dự đoán đúng xác suất thật — không thể "ăn gian" bằng cách nói tự tin hơn hay kém hơn. Accuracy (ở ngưỡng 0.5) không có tính chất này.

**Phân rã Murphy** (với dự đoán gom theo bin $k$, $n_k$ mẫu, dự đoán TB $\bar p_k$, tỉ lệ thật $\bar o_k$, tỉ lệ chung $\bar o$):

$$\text{Brier} \approx \underbrace{\sum_k\tfrac{n_k}{n}(\bar p_k - \bar o_k)^2}_{\text{reliability (càng nhỏ càng tốt)}} - \underbrace{\sum_k\tfrac{n_k}{n}(\bar o_k - \bar o)^2}_{\text{resolution (càng lớn càng tốt)}} + \underbrace{\bar o(1 - \bar o)}_{\text{uncertainty (của dữ liệu)}}$$

Reliability = sai lệch calibration; resolution = khả năng **phân biệt** các nhóm có tỉ lệ khác nhau (liên quan tới xếp hạng); uncertainty không phụ thuộc mô hình.

In [ ]:
def brier_decomposition(p, y, n_bins=10):
    r = reliability(p, y, n_bins)
    w = r["count"] / r["count"].sum()
    rel = np.sum(w * (r.mean_pred - r.frac_pos) ** 2)
    res = np.sum(w * (r.frac_pos - y.mean()) ** 2)
    unc = y.mean() * (1 - y.mean())
    return rel, res, unc


print(f"{'mô hình':20s} {'reliability':>12s} {'resolution':>11s} {'uncertainty':>12s} {'REL−RES+UNC':>12s} {'Brier':>8s}")
for name, p in [*probs.items(), ("xác suất thật", pt_test)]:
    rel, res, unc = brier_decomposition(p, y_test)
    print(f"{name:20s} {rel:12.4f} {res:11.4f} {unc:12.4f} {rel - res + unc:12.4f} {brier_score_loss(y_test, p):8.4f}")

Phân rã gần khớp Brier (sai khác nhỏ do biến thiên **trong** từng bin). Naive Bayes và MLP overfit có reliability lớn (sai lệch calibration) — sửa được bằng calibration hậu kiểm (bài 02) mà **không** cần train lại; còn resolution chỉ cải thiện được bằng mô hình/đặc trưng tốt hơn.

## 6. Vì sao calibration quan trọng: quyết định theo chi phí

Chiến dịch giữ chân: gửi ưu đãi tốn **50** nghìn đồng/khách; nếu khách đang định rời bỏ và được giữ lại, công ty tránh được thiệt hại **200** nghìn đồng (giả sử ưu đãi luôn giữ được khách — chương 38 sẽ xét tác động nhân quả thật). Gửi ưu đãi khi kỳ vọng có lời: $\hat p\cdot 200 > 50 \Leftrightarrow \hat p > 0.25$. Ngưỡng này **chỉ đúng nếu $\hat p$ calibrated**. Lợi nhuận kỳ vọng thật tính bằng $p^*$:

In [ ]:
VALUE, COST = 200, 50
threshold = COST / VALUE
print(f"ngưỡng ra quyết định = {threshold:.2f}")
for name, p in [*probs.items(), ("xác suất thật", pt_test)]:
    send = p > threshold
    profit = np.sum(pt_test[send] * VALUE - COST)
    print(f"{name:20s}: gửi cho {send.mean():5.1%} khách | lợi nhuận kỳ vọng thật = {profit / 1000:8.1f} triệu đồng")

Cùng ngưỡng 0.25, mô hình calibrated (logistic) đạt gần mức tối ưu của xác suất thật; mô hình quá tự tin gửi ưu đãi sai đối tượng — mất tiền dù AUC không tệ. Với mô hình chưa calibrated, hoặc phải calibrate (bài 02), hoặc phải dò lại ngưỡng trên dữ liệu validation theo chính lợi nhuận.

## 🧠 Kiểm tra nhanh

1. Mô hình calibrated nghĩa là gì? Vì sao AUC không đo calibration?
2. Vẽ reliability diagram và tính ECE thế nào? Vì sao ECE của xác suất thật vẫn khác 0?
3. Proper scoring rule là gì? Cho hai ví dụ.
4. Phân rã Brier gồm những thành phần nào? Thành phần nào sửa được bằng calibration hậu kiểm?
5. Vì sao Naive Bayes và mạng nơ-ron overfit thường quá tự tin?
6. Vì sao xác suất sai lệch dẫn tới quyết định sai dù AUC tốt?

## 📌 Tóm tắt

- Calibration: dự đoán $p$ ↔ tần suất thật $p$; độc lập với xếp hạng (AUC).
- Reliability diagram, ECE (có sàn nhiễu do mẫu hữu hạn), Brier, log-loss (proper scoring rules); Brier = reliability − resolution + uncertainty.
- Logistic regression đúng dạng: calibrated; Naive Bayes, MLP overfit: quá tự tin; rừng: thường kém tự tin ở hai đầu.
- Quyết định theo chi phí cần xác suất calibrated.

## 📚 Tài liệu tham khảo

- Niculescu-Mizil & Caruana (2005) — *Predicting Good Probabilities With Supervised Learning*
- Guo et al. (2017) — *On Calibration of Modern Neural Networks*
- Murphy (1973) — *A New Vector Partition of the Probability Score*
- Gneiting & Raftery (2007) — *Strictly Proper Scoring Rules, Prediction, and Estimation*
- [scikit-learn — Probability calibration](https://scikit-learn.org/stable/modules/calibration.html)